# 06_Scikit_Learn.ipynb

---

# 1. Estimator

Scikit-learn provides two KNN estimators:

| Estimator              | Purpose        |
| ---------------------- | -------------- |
| `KNeighborsClassifier` | Classification |
| `KNeighborsRegressor`  | Regression     |

For this notebook, we'll primarily focus on **`KNeighborsClassifier`**. The regressor works similarly, except it predicts the **average** of the neighbors instead of the majority class.

---

# 2. Import & Constructor

## Required Imports

```python
import numpy as np

from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split

from sklearn.preprocessing import StandardScaler

from sklearn.neighbors import KNeighborsClassifier

from sklearn.metrics import accuracy_score
```

---

## Constructor Syntax

```python
KNeighborsClassifier(
    n_neighbors=5,
    weights='uniform',
    algorithm='auto',
    leaf_size=30,
    p=2,
    metric='minkowski',
    metric_params=None,
    n_jobs=None
)
```

---

## Parameter Table

| Parameter       | Default       | Description                            | Common Usage                                |
| --------------- | ------------- | -------------------------------------- | ------------------------------------------- |
| **n_neighbors** | `5`           | Number of nearest neighbors (K)        | `3`, `5`, `7`, `9`                          |
| **weights**     | `"uniform"`   | How neighbors contribute               | `"uniform"` or `"distance"`                 |
| **algorithm**   | `"auto"`      | Neighbor search algorithm              | Leave as `"auto"`                           |
| **leaf_size**   | `30`          | Tree leaf size (KDTree/BallTree)       | Usually default                             |
| **p**           | `2`           | Power parameter for Minkowski distance | `1` = Manhattan, `2` = Euclidean            |
| **metric**      | `"minkowski"` | Distance metric                        | `"minkowski"`, `"euclidean"`, `"manhattan"` |
| **n_jobs**      | `None`        | Number of CPU cores                    | `-1` to use all cores                       |

---

## Parameter Explanation

### 1. `n_neighbors`

Controls **K**.

Example:

```python
KNeighborsClassifier(n_neighbors=3)
```

Small K

* More flexible
* Sensitive to noise
* High variance

Large K

* Smoother boundary
* Less sensitive to noise
* High bias

---

### 2. `weights`

Determines how neighbors vote.

#### Uniform (Default)

Every neighbor has equal importance.

```python
weights="uniform"
```

Example:

| Distance | Class |
| -------- | ----- |
| 1        | Dog   |
| 2        | Dog   |
| 8        | Cat   |

Votes

```text
Dog = 2

Cat = 1
```

Prediction:

```text
Dog
```

---

#### Distance

Closer neighbors receive higher importance.

```python
weights="distance"
```

Example

| Distance | Class |
| -------- | ----- |
| 1        | Dog   |
| 2        | Cat   |
| 8        | Cat   |

Although Cat has two neighbors, the closest Dog may receive a larger weight, changing the prediction.

Use `"distance"` when closer samples should influence the prediction more strongly.

---

### 3. `algorithm`

Controls **how nearest neighbors are searched**.

Options:

```python
"auto"
"ball_tree"
"kd_tree"
"brute"
```

#### `"auto"` (Recommended)

Scikit-learn automatically selects the most suitable search algorithm based on the dataset.

This is what you should use in almost every project.

---

#### `"brute"`

Computes the distance from the query point to **every** training sample.

```text
Query
   ↓
Distance to Sample 1
Distance to Sample 2
Distance to Sample 3
...
Distance to Sample n
```

* Simple
* Good for small datasets
* Slower on large datasets

---

#### `"kd_tree"`

Builds a **KD-Tree**, a binary tree that partitions the feature space.

Instead of checking every point, it can eliminate large regions that cannot contain the nearest neighbor.

Advantages:

* Faster for low-dimensional numerical data.
* Commonly effective when the number of features is relatively small.

---

#### `"ball_tree"`

Builds a **Ball Tree**, where points are grouped into hyperspherical regions ("balls").

Advantages:

* Often performs better than KD-Tree on higher-dimensional datasets.
* Supports a wider range of distance metrics efficiently.

---

### Should You Choose One?

Usually **No**.

Use:

```python
algorithm="auto"
```

Scikit-learn chooses among `brute`, `kd_tree`, and `ball_tree` based on:

* Dataset size
* Number of features
* Distance metric

---

### 4. `leaf_size`

Used only by KDTree and BallTree.

```python
leaf_size=30
```

It determines how many samples are stored in each leaf of the tree.

Effects:

* Smaller leaf size:

  * Larger tree
  * Potentially faster queries
  * More memory

* Larger leaf size:

  * Smaller tree
  * Less memory
  * Sometimes slower queries

For most datasets:

```python
leaf_size=30
```

is a good default.

---

### 5. `metric`

Specifies the distance metric.

Examples:

```python
metric="euclidean"
```

```python
metric="manhattan"
```

```python
metric="minkowski"
```

The default is:

```python
metric="minkowski"
```

Combined with:

```python
p=2
```

this becomes **Euclidean Distance**.

---

### 6. `p`

Only relevant when:

```python
metric="minkowski"
```

* `p=1` → Manhattan Distance
* `p=2` → Euclidean Distance

Examples:

```python
KNeighborsClassifier(metric="minkowski", p=1)
```

uses Manhattan Distance.

```python
KNeighborsClassifier(metric="minkowski", p=2)
```

uses Euclidean Distance.

---

### 7. `n_jobs`

Controls parallel processing.

```python
n_jobs=-1
```

Use all available CPU cores.

Useful for:

* Large datasets
* Cross-validation
* GridSearchCV

---

## Recommended Settings

For most classification problems:

```python
model = KNeighborsClassifier(
    n_neighbors=5,
    weights="uniform",
    metric="minkowski",
    p=2,
    algorithm="auto"
)
```

If the dataset is noisy, try:

```python
weights="distance"
```

If the dataset is large, use:

```python
n_jobs=-1
```

---

## Best Practices

* **Always scale numerical features** before using KNN.
* Start with **`n_neighbors=5`** and tune it using cross-validation.
* Use `algorithm="auto"` unless you have a specific reason to choose another algorithm.
* Experiment with both `weights="uniform"` and `weights="distance"`.
* Use `GridSearchCV` to tune `n_neighbors`, `weights`, and `metric`.
* KNN performs best on **small to medium-sized datasets**. Prediction can become slow on very large datasets.

---

# 3. Methods & Attributes

## Methods Table

| Method             | Purpose                                                  | Returns                      |
| ------------------ | -------------------------------------------------------- | ---------------------------- |
| `fit(X, y)`        | Store the training data and prepare the search structure | Estimator                    |
| `predict(X)`       | Predict class labels                                     | NumPy array                  |
| `predict_proba(X)` | Return class probabilities                               | NumPy array                  |
| `score(X, y)`      | Return mean accuracy                                     | Float                        |
| `kneighbors(X)`    | Return distances and indices of nearest neighbors        | Tuple `(distances, indices)` |

---

## Attributes Table

| Attribute                  | Description                                                              |
| -------------------------- | ------------------------------------------------------------------------ |
| `classes_`                 | Unique class labels seen during training                                 |
| `effective_metric_`        | Distance metric actually used                                            |
| `effective_metric_params_` | Parameters of the metric used                                            |
| `n_features_in_`           | Number of input features                                                 |
| `feature_names_in_`        | Feature names (if training data is a DataFrame with string column names) |
| `n_samples_fit_`           | Number of training samples stored                                        |

---

## One Complete Code Example

```python
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score

# Load data
iris = load_iris()

X = iris.data
y = iris.target

# Split data
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# Scale features
scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Create model
model = KNeighborsClassifier(
    n_neighbors=5
)

# Train
model.fit(X_train, y_train)

# Predict
y_pred = model.predict(X_test)

# Accuracy
print(accuracy_score(y_test, y_pred))
```

---

## Commonly Used Methods

### `fit()`

Stores the training data and builds the internal neighbor search structure.

```python
model.fit(X_train, y_train)
```

---

### `predict()`

Predicts the class labels.

```python
predictions = model.predict(X_test)
```

---

### `predict_proba()`

Returns class probabilities.

```python
prob = model.predict_proba(X_test)
```

Example output:

```python
[[0.0, 0.1, 0.9]]
```

Meaning:

* Class 0 → 0%
* Class 1 → 10%
* Class 2 → 90%

---

### `score()`

Returns the mean accuracy.

```python
accuracy = model.score(X_test, y_test)
```

Equivalent to:

```python
accuracy_score(y_test, model.predict(X_test))
```

---

### `kneighbors()`

One of the most useful methods for understanding KNN.

```python
distances, indices = model.kneighbors(X_test)
```

Returns:

* Distances to the nearest neighbors.
* Indices of those neighbors in the training set.

Example:

```python
distances
```

```python
[[0.32, 0.48, 0.61]]
```

```python
indices
```

```python
[[15, 42, 8]]
```

This means the nearest neighbors are training samples at indices `15`, `42`, and `8`.

---

## Commonly Used Attributes

```python
model.classes_
```

Output:

```python
array([0, 1, 2])
```

---

```python
model.n_features_in_
```

Output:

```python
4
```

---

```python
model.n_samples_fit_
```

Output:

```python
120
```

---

```python
model.effective_metric_
```

Output:

```python
'minkowski'
```

---

# 4. End-to-End Workflow

## Workflow Diagram

```text
Load Data
    ↓
Train-Test Split
    ↓
Feature Scaling
    ↓
Train KNN
    ↓
Predict
    ↓
Evaluate
```

---

## One Complete Working Example

```python
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier

# Load data
X, y = load_iris(return_X_y=True)

# Split
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# Scale
scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Model
model = KNeighborsClassifier(
    n_neighbors=5
)

# Train
model.fit(X_train, y_train)

# Predict
predictions = model.predict(X_test)

# Accuracy
print(model.score(X_test, y_test))
```

---

## Important Notes

* **Always fit the scaler only on the training data**, then transform both training and test data.
* Scaling is much more important for KNN than for tree-based algorithms.
* Use `stratify=y` for classification to preserve class proportions in the train/test split.
* `fit()` does **not** learn weights—it stores the training data and prepares the neighbor search structure.

---

## Common Errors

| Mistake                                   | Why It Happens                                                   |
| ----------------------------------------- | ---------------------------------------------------------------- |
| Not scaling the data                      | Features with larger values dominate the distance calculation.   |
| Using a very small K (e.g., `1`)          | Model becomes highly sensitive to noise (high variance).         |
| Using a very large K                      | Decision boundary becomes too smooth (high bias).                |
| Fitting the scaler on the full dataset    | Causes **data leakage** and overly optimistic evaluation.        |
| Choosing an inappropriate distance metric | The notion of "nearest" may no longer reflect true similarity.   |
| Using KNN on very large datasets          | Prediction becomes slow because many distances must be computed. |

---